# Global registration with RANSAC
We are going to use [open3d](http://www.open3d.org/) to handle point clouds and generation of point clouds.
We are importing the packages and defining a function which helps us drawing the point clouds.

In [7]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    # We use deepcopy to ensure we don't modify the original point cloud data in memory
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    
    # If recolor is True, paint the point clouds in distinct colors (Orange vs Cyan)
    # This helps visually distinguish the two clouds to check alignment.
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    
    # If a transformation matrix (4x4) is provided, apply it to the source cloud
    if(transformation is not None):
        source_temp.transform(transformation)
        
    # Open the visualizer window
    o3d.visualization.draw_geometries([source_temp, target_temp])

We need to read in our pointclouds. For that we use the `io` module of the
open3d package (`o3d`). The following cell will open a window with a
visualization of both point clouds `source` and `target`.

Also, this page [Visualization - Open3D](http://open3d.org/html/tutorial/Basic/visualization.html)
contains some useful examples and instructions on how to use the viewer.

In [8]:
path = "C:/Users/peisz/OneDrive/Documents/Egyetem/DTU/3rd semester/PAS/Week 5/"
source = o3d.io.read_point_cloud(path + "ICP/r1.pcd")
target = o3d.io.read_point_cloud(path + "ICP/r2.pcd")

# Used for downsampling.
# This defines the resolution of the voxel grid (5cm in this case, assuming units are meters).
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

### Finding features in pointclouds
When working on point clouds it can be beneficial to work on a downsampled version of the point cloud,
as it decreases the need of computation.

You can use [`pointcloud.voxel_down_sample()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample) where `pointcloud` is the name of your point cloud object. In our case, that would be `source` and `target`.

We also need to estimate the normals of the point cloud points using [`pointcloud.estimate_normals()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.estimate_normals)

**Task:** Find FPFH features or correspondances of the downsampled point clouds.
[`o3d.pipelines.registration.compute_fpfh_feature()`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.compute_fpfh_feature.html)


In [9]:
####
# Downsample and find features here
####

# Voxel Downsampling:
# Creates a 3D grid (voxels) of the specified size. All points falling into one voxel are averaged into a single point.
# This significantly reduces the number of points while preserving the overall geometry.
source_down = source.voxel_down_sample(voxel_size)
target_down = target.voxel_down_sample(voxel_size)

# Normal Estimation:
# Computes the surface normal vector for every point.
# This is required for:
# 1. Feature extraction (FPFH relies on the relationship between neighbors' normals).
# 2. Point-to-Plane registration (uses the dot product of the normal and the distance vector).
source_down.estimate_normals()
target_down.estimate_normals()

# Compute FPFH (Fast Point Feature Histograms) features.
# FPFH is a 33-dimensional vector that describes the local geometric structure around a point.
# It is invariant to pose (rotation/translation), meaning the feature vector for a corner of a table
# should look the same regardless of how the table is rotated.
#
# o3d.geometry.KDTreeSearchParamHybrid(radius, max_nn):
# Defines the neighborhood search. It looks for up to 'max_nn' neighbors within 'radius'.
# Here we use a radius of 5x the voxel size to capture sufficient local context.
source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(source_down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(target_down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))


### RANSAC 
We will now attempt to use RANSAC to do a global registration of the two point clouds.

By using the function [`o3d.pipelines.registration.registration_ransac_based_on_feature_matching`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.registration_ransac_based_on_feature_matching.html) from open3d, do the following:


Try to find the transformation from `r1.pcd` (`source`) to `r2.pcd` (`target`).
Attempt with point-to-point and point-to-plane
```Python
point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()
```

When using RANSAC, focus on the arguments below. The rest are optional parameters.
```Python
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point)
```

In [10]:
####
# Call RANSAC here
####

# Define the estimation methods
# PointToPoint: minimizes the squared distance between corresponding points.
# PointToPlane: minimizes the distance between a source point and the tangent plane of the target point. 
# (PointToPlane is generally more robust for flat surfaces but requires good normals).
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

# Distance threshold: Max distance between a source point and a target point to be considered an "inlier".
distance_threshold = voxel_size * 1.5

# RANSAC (Random Sample Consensus) Global Registration:
# 1. Randomly picks 3 points from source.
# 2. Finds their corresponding points in target based on FPFH feature similarity.
# 3. Estimates a transformation.
# 4. Validates the transform (checks if enough points align).
# 5. Repeats for millions of iterations and keeps the best result.
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down, 
    source_fpfh, target_fpfh, 
    True, # Mutual filter: ensures correspondence is bidirectional (A matches B AND B matches A)
    distance_threshold,
    point_to_point)

draw_registrations(source, target, ransac_result.transformation, True)

## Exercises
### A)
Can you get a decent transformation from r1 to r3? (check the ICP folder)


In [11]:
targetNew = o3d.io.read_point_cloud(path + "ICP/r3.pcd")

# Note: Ideally you should downsample 'targetNew', but here 'target' is used. 
# Make sure to check if you intended to use 'targetNew.voxel_down_sample'.
targetNew_down = targetNew.voxel_down_sample(voxel_size) 
targetNew_fpfh = o3d.pipelines.registration.compute_fpfh_feature(targetNew_down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))

ransacNew_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, targetNew_down, 
    source_fpfh, targetNew_fpfh, 
    False, # Mutual filter disabled here
    distance_threshold,
    point_to_point)

draw_registrations(source, targetNew, ransacNew_result.transformation, True)

### B)
With the following checkers, can you get better results from RANSAC? Try tweaking the parameters of them. Can you make point-to-plane work? Do not spend too much time on this, if you can't manage, skip it. (I was not able to get a good fit.)

You can also try tweaking the `voxel_size`

```Python
corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)
```

In [12]:
# Correspondence Checkers allow us to reject bad transformations early in the RANSAC loop.
corr_length = 0.9
distance_threshold = voxel_size * 1.5

# Checker 1: Edge Length
# Checks if the distance between any two points in the source is similar to the distance 
# between the corresponding points in the target. (i.e., rigid body constraint).
c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)

# Checker 2: Distance
# Checks if the aligned points are close enough to each other.
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)

# Checker 3: Normal
# Checks if the normals of corresponding points are similar (dot product close to 1).
# 0.095 is the cosine of the max allowed angle difference (~84 degrees).
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransacCheck_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)

draw_registrations(source, target, ransacCheck_result.transformation, True)